In [ ]:
import pickle

# Load pickle file
with open("gradient_test_exponax_nu0.01_nsamples1.pkl", "rb") as f:
    data = pickle.load(f)

: 

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import imageio
import os

def generate_gif_from_data(data, output_dir="gifs", num_steps=100):
    os.makedirs(output_dir, exist_ok=True)

    for key, content in data.items():
        frames = []
        loss_history = {"with_solver": [], "without_solver": [], "approximate": []}

        for step_idx in range(num_steps):
            step_key = f"step_{step_idx}"
            if step_key not in content["step_update"]:
                continue
            
            step_data = content["step_update"][step_key]
            values = step_data["values"]

            # 累积 loss
            loss_vals = step_data["loss_metrics"]
            for loss_type in loss_history:
                loss_history[loss_type].append(loss_vals[loss_type])

            # 创建画布
            fig, axs = plt.subplots(1, 5, figsize=(25, 5))

            # (1) 初始状态对比
            axs[0].plot(values["a_values"]["a"], label="a")
            axs[0].plot(values["a_values"]["a_with"], label="a_with")
            axs[0].plot(values["a_values"]["a_without"], label="a_without")
            axs[0].plot(values["a_values"]["a_approximate"], label="a_approximate")
            axs[0].set_title(f"Step {step_idx} - Initial states")
            axs[0].legend()

            # (2) 扰动方法1
            axs[1].plot(values["G_values"]["G(a)"], label="G(a) (FNO)")
            axs[1].plot(values["g_values"]["g(a)"], label="g(a) (PDE)")
            axs[1].set_title("With Solver")
            axs[1].legend()

            # (3) 扰动方法2
            axs[2].plot(values["G_values"]["G(a_without)"], label="G(a_without) (FNO)")
            axs[2].plot(values["g_values"]["g(a_without)"], label="g(a_without) (PDE)")
            axs[2].set_title("Without Solver")
            axs[2].legend()

            # (4) 扰动方法3
            axs[3].plot(values["G_values"]["G(a_approximate)"], label="G(a_approximate) (FNO)")
            axs[3].plot(values["g_values"]["g(a_approximate)"], label="g(a_approximate) (PDE)")
            axs[3].set_title("Approximate")
            axs[3].legend()

            # (5) Loss 曲线
            axs[4].plot(loss_history["with_solver"], label="with_solver")
            axs[4].plot(loss_history["without_solver"], label="without_solver")
            axs[4].plot(loss_history["approximate"], label="approximate")
            axs[4].set_title("Loss History")
            axs[4].legend()

            plt.tight_layout()

            # 保存临时帧
            frame_path = os.path.join(output_dir, f"temp_{step_idx}.png")
            plt.savefig(frame_path)
            plt.close(fig)
            frames.append(imageio.imread(frame_path))

        # 保存 GIF
        gif_path = os.path.join(output_dir, f"{'_'.join(map(str,key))}.gif")
        imageio.mimsave(gif_path, frames, fps=5)

        # 删除临时帧
        for step_idx in range(num_steps):
            temp_path = os.path.join(output_dir, f"temp_{step_idx}.png")
            if os.path.exists(temp_path):
                os.remove(temp_path)

        print(f"GIF saved: {gif_path}")


# 读取 pickle
with open("gradient_test_exponax_nu0.01_nsamples1.pkl", "rb") as f:
    data = pickle.load(f)

# 生成 GIF
generate_gif_from_data(data, output_dir="gifs", num_steps=100)
